# EthicaAI — Melting Pot Benchmark
**Commitment Floors for Tipping-Point Commons**

This notebook runs the Melting Pot external benchmark validation.
- Substrates: `commons_harvest__open`, `clean_up`
- Policies: Selfish, Commitment (φ₁=0.5,0.7,1.0), IPPO baseline, IPPO+Floor
- 20 seeds per condition

**Runtime: GPU (T4 or A100)**

In [ ]:
# Cell 1: Install Melting Pot (REQUIRES Python 3.10)
# *** Runtime > Change runtime type > Python 3.10 + GPU ***
import sys
print(f"Python: {sys.version}")
if sys.version_info[:2] != (3, 10):
    print("ERROR: Need Python 3.10!")
    print("Go to: Runtime > Change runtime type > Python 3.10")
    raise SystemExit("Switch to Python 3.10")

!pip install dmlab2d==1.0.0 -q
!pip install dm-meltingpot -q
!pip install torch -q

import dmlab2d, meltingpot
print(f"OK: Melting Pot ready")

In [ ]:
# Cell 2: Environment Test
from meltingpot import substrate
import numpy as np

for sub_name in ['commons_harvest__open', 'clean_up']:
    env_config = substrate.get_config(sub_name)
    env = substrate.build(env_config)
    ts = env.reset()
    n_agents = len(ts.observation)
    obs_shape = ts.observation[0]['RGB'].shape if 'RGB' in ts.observation[0] else 'unknown'
    print(f'{sub_name}: {n_agents} agents, obs={obs_shape}')
    env.close()

In [ ]:
# Cell 3: Policy Definitions
import numpy as np
import json
import time
from meltingpot import substrate

N_SEEDS = 20
N_STEPS = 500
SUBSTRATES = ['commons_harvest__open', 'clean_up']

def run_episode(substrate_name, policy_fn, seed=0):
    rng = np.random.RandomState(seed)
    env_config = substrate.get_config(substrate_name)
    env = substrate.build(env_config)
    timestep = env.reset()
    n_agents = len(timestep.observation)
    total_reward = np.zeros(n_agents)
    steps = 0
    for t in range(N_STEPS):
        actions = {}
        for i in range(n_agents):
            actions[i] = policy_fn(timestep.observation[i], i, rng, t)
        timestep = env.step(actions)
        for i in range(n_agents):
            total_reward[i] += timestep.reward[i]
        steps += 1
        if timestep.last():
            break
    env.close()
    return {'total_reward': float(np.mean(total_reward)), 'steps': steps}

def selfish_policy(obs, agent_id, rng, t):
    return rng.randint(0, 8)

def make_commitment_policy(phi):
    """Floor policy: P(interact) >= phi, P(move) = 1-phi."""
    def policy(obs, agent_id, rng, t):
        if rng.random() < phi:
            return rng.choice([5, 6, 7])  # interact
        else:
            return rng.choice([0, 1, 2, 3, 4])  # move
    return policy

print('Policies defined.')

In [ ]:
# Cell 4: Run Heuristic Experiments
t0 = time.time()
results = {}

policies = {
    'Selfish': selfish_policy,
    'Floor_0.3': make_commitment_policy(0.3),
    'Floor_0.5': make_commitment_policy(0.5),
    'Floor_0.7': make_commitment_policy(0.7),
    'Floor_1.0': make_commitment_policy(1.0),
}

for sub_name in SUBSTRATES:
    print(f'\n=== {sub_name} ===')
    results[sub_name] = {}
    
    for pol_name, pol_fn in policies.items():
        print(f'  [{pol_name}]', end='', flush=True)
        rewards = []
        for s in range(N_SEEDS):
            try:
                r = run_episode(sub_name, pol_fn, seed=42+s)
                rewards.append(r['total_reward'])
            except Exception as e:
                print(f' ERROR:{e}', end='')
                rewards.append(0.0)
            if (s+1) % 10 == 0:
                print(f' [{s+1}]', end='', flush=True)
        
        mean_r = float(np.mean(rewards))
        std_r = float(np.std(rewards))
        results[sub_name][pol_name] = {
            'mean': round(mean_r, 2),
            'std': round(std_r, 2),
            'rewards': [round(r, 2) for r in rewards],
        }
        print(f'  => {mean_r:.1f} +/- {std_r:.1f}')

elapsed = time.time() - t0
print(f'\nTotal: {elapsed:.0f}s')

In [ ]:
# Cell 5: IPPO Training with CNN (GPU)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

class CNNPolicy(nn.Module):
    def __init__(self, n_actions=8):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 16, 8, stride=4),
            nn.ReLU(),
            nn.Conv2d(16, 32, 4, stride=2),
            nn.ReLU(),
            nn.Flatten(),
        )
        # Compute conv output size
        with torch.no_grad():
            dummy = torch.zeros(1, 3, 88, 88)
            conv_out = self.conv(dummy).shape[1]
        self.fc = nn.Sequential(
            nn.Linear(conv_out, 256),
            nn.ReLU(),
        )
        self.actor = nn.Linear(256, n_actions)
        self.critic = nn.Linear(256, 1)
    
    def forward(self, x):
        # x: (B, H, W, C) uint8 -> (B, C, H, W) float
        if x.dim() == 3:
            x = x.unsqueeze(0)
        x = x.permute(0, 3, 1, 2).float() / 255.0
        feat = self.fc(self.conv(x))
        return self.actor(feat), self.critic(feat)
    
    def get_action(self, obs, floor_actions=None, floor_prob=0.0):
        logits, value = self.forward(obs)
        # Apply floor: boost probability of cooperative actions
        if floor_prob > 0 and floor_actions is not None:
            probs = torch.softmax(logits, dim=-1)
            # Redistribute: give floor_prob to cooperative actions
            boost = torch.zeros_like(probs)
            boost[:, floor_actions] = floor_prob / len(floor_actions)
            probs = (1 - floor_prob) * probs + boost
            dist = Categorical(probs=probs)
        else:
            dist = Categorical(logits=logits)
        action = dist.sample()
        return action, dist.log_prob(action), value.squeeze(-1)

print('CNNPolicy defined.')

In [ ]:
# Cell 6: IPPO Training Loop
TRAIN_EPISODES = 100  # Reduced for Colab time limits
EVAL_EPISODES = 20
LR = 3e-4
GAMMA = 0.99
CLIP_EPS = 0.2
INTERACT_ACTIONS = [5, 6, 7]  # Cooperative actions in Melting Pot

def train_ippo(substrate_name, floor_prob=0.0, seed=0, verbose=True):
    """Train IPPO agents on a Melting Pot substrate."""
    rng = np.random.RandomState(seed)
    env_config = substrate.get_config(substrate_name)
    env = substrate.build(env_config)
    ts = env.reset()
    n_agents = len(ts.observation)
    env.close()
    
    # One policy per agent (IPPO = independent)
    policies = [CNNPolicy().to(device) for _ in range(n_agents)]
    optimizers = [optim.Adam(p.parameters(), lr=LR) for p in policies]
    
    floor_act = torch.tensor(INTERACT_ACTIONS, device=device)
    
    train_rewards = []
    
    for ep in range(TRAIN_EPISODES):
        env = substrate.build(env_config)
        ts = env.reset()
        
        ep_data = {i: {'obs': [], 'acts': [], 'logp': [], 'vals': [], 'rews': []} 
                   for i in range(n_agents)}
        ep_reward = 0
        
        for t in range(200):  # Shorter horizon for speed
            actions = {}
            for i in range(n_agents):
                obs_rgb = ts.observation[i].get('RGB', np.zeros((88,88,3), dtype=np.uint8))
                obs_t = torch.tensor(obs_rgb, device=device, dtype=torch.uint8)
                
                with torch.no_grad():
                    act, logp, val = policies[i].get_action(
                        obs_t, floor_actions=floor_act, floor_prob=floor_prob)
                
                actions[i] = int(act.item())
                ep_data[i]['obs'].append(obs_t)
                ep_data[i]['acts'].append(act)
                ep_data[i]['logp'].append(logp)
                ep_data[i]['vals'].append(val)
            
            ts = env.step(actions)
            
            for i in range(n_agents):
                ep_data[i]['rews'].append(ts.reward[i])
                ep_reward += ts.reward[i]
            
            if ts.last():
                break
        
        env.close()
        train_rewards.append(ep_reward / n_agents)
        
        # PPO update for each agent
        for i in range(n_agents):
            if len(ep_data[i]['rews']) < 2:
                continue
            
            rewards = torch.tensor(ep_data[i]['rews'], device=device, dtype=torch.float32)
            old_logp = torch.stack(ep_data[i]['logp'])
            old_vals = torch.stack(ep_data[i]['vals'])
            
            # Compute returns
            returns = torch.zeros_like(rewards)
            G = 0
            for t in reversed(range(len(rewards))):
                G = rewards[t] + GAMMA * G
                returns[t] = G
            
            advantages = returns - old_vals.detach()
            advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
            
            # PPO loss
            obs_batch = torch.stack(ep_data[i]['obs'])
            act_batch = torch.stack(ep_data[i]['acts'])
            
            logits, values = policies[i](obs_batch)
            if floor_prob > 0:
                probs = torch.softmax(logits, dim=-1)
                boost = torch.zeros_like(probs)
                boost[:, INTERACT_ACTIONS] = floor_prob / len(INTERACT_ACTIONS)
                probs = (1 - floor_prob) * probs + boost
                dist = Categorical(probs=probs)
            else:
                dist = Categorical(logits=logits)
            
            new_logp = dist.log_prob(act_batch)
            ratio = torch.exp(new_logp - old_logp.detach())
            
            surr1 = ratio * advantages
            surr2 = torch.clamp(ratio, 1-CLIP_EPS, 1+CLIP_EPS) * advantages
            
            actor_loss = -torch.min(surr1, surr2).mean()
            critic_loss = 0.5 * (returns - values.squeeze(-1)).pow(2).mean()
            entropy = dist.entropy().mean()
            
            loss = actor_loss + critic_loss - 0.01 * entropy
            
            optimizers[i].zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(policies[i].parameters(), 0.5)
            optimizers[i].step()
        
        if verbose and (ep+1) % 10 == 0:
            print(f'  Ep {ep+1}/{TRAIN_EPISODES}: reward={train_rewards[-1]:.1f}')
    
    return policies, train_rewards

print('Training function defined.')

In [ ]:
# Cell 7: Evaluate Trained IPPO
def evaluate_ippo(substrate_name, policies, floor_prob=0.0, n_eval=20):
    env_config = substrate.get_config(substrate_name)
    n_agents = len(policies)
    floor_act = torch.tensor(INTERACT_ACTIONS, device=device)
    
    rewards = []
    for s in range(n_eval):
        env = substrate.build(env_config)
        ts = env.reset()
        ep_reward = 0
        for t in range(N_STEPS):
            actions = {}
            for i in range(n_agents):
                obs_rgb = ts.observation[i].get('RGB', np.zeros((88,88,3), dtype=np.uint8))
                obs_t = torch.tensor(obs_rgb, device=device, dtype=torch.uint8)
                with torch.no_grad():
                    act, _, _ = policies[i].get_action(
                        obs_t, floor_actions=floor_act, floor_prob=floor_prob)
                actions[i] = int(act.item())
            ts = env.step(actions)
            for i in range(n_agents):
                ep_reward += ts.reward[i]
            if ts.last():
                break
        env.close()
        rewards.append(ep_reward / n_agents)
    
    return {'mean': round(float(np.mean(rewards)), 2),
            'std': round(float(np.std(rewards)), 2),
            'rewards': [round(r, 2) for r in rewards]}

In [ ]:
# Cell 8: Full Experiment — Focus on clean_up (1000 episodes, GPU)
import torch
ippo_results = {}
FLOOR_PROBS = [0.0, 0.2, 0.4]
N_TRAIN_SEEDS = 3
TRAIN_EPISODES = 1000  # Long training for clean_up
INTERACT_ACTIONS = [5, 6, 7]
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

for sub_name in ['clean_up']:
    print(f'\n{"="*60}')
    print(f'  IPPO on {sub_name} (1000 episodes)')
    print(f'{"="*60}')
    ippo_results[sub_name] = {}
    
    for floor_prob in FLOOR_PROBS:
        print(f'\n  --- Floor prob = {floor_prob} ---')
        all_eval = []
        
        for seed in range(N_TRAIN_SEEDS):
            print(f'  Training seed {seed+1}/{N_TRAIN_SEEDS}...')
            policies, train_r = train_ippo(sub_name, floor_prob=floor_prob, 
                                           seed=42+seed, verbose=True)
            eval_r = evaluate_ippo(sub_name, policies, floor_prob=floor_prob, n_eval=10)
            all_eval.append(eval_r['mean'])
            print(f'    Eval reward: {eval_r["mean"]:.1f}')
        
        ippo_results[sub_name][f'IPPO_floor_{floor_prob}'] = {
            'mean': round(float(np.mean(all_eval)), 2),
            'std': round(float(np.std(all_eval)), 2),
            'per_seed': [round(r, 2) for r in all_eval],
        }
        print(f'  IPPO (floor={floor_prob}): {np.mean(all_eval):.1f} +/- {np.std(all_eval):.1f}')

# Also run heuristic on clean_up for comparison
print('\n--- Heuristic baselines on clean_up ---')
for pol_name, pol_fn in [("Selfish", selfish_policy), 
                          ("Floor_0.5", make_commitment_policy(0.5)),
                          ("Floor_1.0", make_commitment_policy(1.0))]:
    rews = [run_episode('clean_up', pol_fn, seed=42+s)['total_reward'] for s in range(10)]
    print(f'  {pol_name}: {np.mean(rews):.1f} +/- {np.std(rews):.1f}')

print('\nIPPO experiments complete!')

In [ ]:
# Cell 9: Combine All Results & Save
final_results = {
    'experiment': 'EthicaAI Melting Pot Benchmark',
    'config': {
        'n_seeds_heuristic': N_SEEDS,
        'n_seeds_ippo': N_TRAIN_SEEDS,
        'n_steps': N_STEPS,
        'train_episodes': TRAIN_EPISODES,
        'substrates': SUBSTRATES,
        'floor_probs': FLOOR_PROBS,
    },
    'heuristic_results': results,
    'ippo_results': ippo_results,
}

# Save
with open('meltingpot_colab_results.json', 'w') as f:
    json.dump(final_results, f, indent=2)

# Print summary table
print('\n' + '='*70)
print('  MELTING POT BENCHMARK RESULTS')
print('='*70)
for sub in SUBSTRATES:
    print(f'\n  {sub}:')
    if sub in results:
        for pol, stats in results[sub].items():
            print(f'    {pol:20s}: {stats["mean"]:8.1f} +/- {stats["std"]:.1f}')
    if sub in ippo_results:
        for pol, stats in ippo_results[sub].items():
            print(f'    {pol:20s}: {stats["mean"]:8.1f} +/- {stats["std"]:.1f}')

# Download
try:
    from google.colab import files
    files.download('meltingpot_colab_results.json')
    print('\nJSON downloaded!')
except:
    print('\nSaved locally: meltingpot_colab_results.json')